
# Cohesive DuckDB Analysis Notebook

### Notebook goals
1. Profile the three core tables and confirm schema assumptions.
2. Quantify data quality issues before building interpretations.
3. Connect engagement activity to technology adoption signals.
4. Investigate anomalies and "weird" patterns worth deeper follow-up.
5. Set up segmentation, feature engineering, and downstream modeling.

### Project framing
The kickoff emphasized that the business question is not just increasing interaction counts, but understanding whether engagement leads to meaningful technology adoption. The requested deliverables include developer cohort profiles, asset impact analysis, a data enrichment plan, and a repeatable framework for future monitoring.



## 1. Environment setup
Update the database path below if your local file lives elsewhere.


In [2]:

import duckdb
import pandas as pd

DB_PATH = "developer_project.duckdb"
con = duckdb.connect(DB_PATH)
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)



## 2. Table inventory and quick orientation
Start by confirming the tables available in the working database.


In [3]:
con.execute("SHOW TABLES").fetchdf()

,name
0,activity_capped
1,activity_clean
2,activity_final
3,activity_raw
4,activity_sample
5,activity_score_bounds
6,activity_score_mapping_clean
7,activity_score_mapping_raw
8,activity_scored
9,activity_stage



## 3. Project hypotheses to test
These hypotheses help connect descriptive analysis to the final deliverables.

- Highly engaged developers are more likely to show downstream adoption signals.
- Some developers may adopt technology without passing through high-touch engagement assets.
- Activity counts alone may overstate impact if repeated or low-value interactions dominate the data.
- Geography, organization quality, and source-system inconsistencies may distort segmentation unless addressed.


## 4. Current baseline analysis already completed

### Activity table baseline

In [4]:

con.execute("SELECT COUNT(*) AS row_count FROM activity_clean").fetchdf()


,row_count
0,69347526


In [5]:

con.execute("SELECT * FROM activity_raw LIMIT 30").fetchdf()


,activity,activity_date,activity_name,activity_type,activity_role,activity_attendance,dev_contact,activity_score,filepath,activity_id,pk1,pk2,lead_source,nvidia_campaign_id,gtc_nvidia_campaign_id,lead_source_details
0,DLI Training,2025-11-12T06:39:30.000Z,Building RAG Agents with LLMs,Instructor-Led,Attendee,Attended Live,6417584,40.0,None,1875418,1875418,course-v1:DLI+S-FX-15+V1-ZH,None,None,None,None
1,DLI Training,2024-05-18T15:53:18.000Z,Getting Started with Deep Learning,Self-Paced Online,Attendee,Attended Live,6752524,20.0,None,764222,764222,course-v1:DLI+S-FX-01+V1,None,None,None,None
2,DLI Training,2024-04-17T09:30:35.000Z,(COURSE ENDING ON 7/1/24) High-Performance Com...,Self-Paced Online,Attendee,Attended Live,6389272,20.0,None,736857,736857,course-v1:DLI+L-AC-25+V1-ZH,None,None,None,None
3,DLI Training,2024-05-22T11:40:34.000Z,Getting Started with Deep Learning,Self-Paced Online,Attendee,Attended Live,6763105,20.0,None,766590,766590,course-v1:DLI+S-FX-01+V1,None,None,None,None
4,DLI Training,2024-05-30T08:31:34.000Z,Prompt Engineering with LLaMA-2 (Access Expire...,Self-Paced Online,Attendee,Attended Live,6790671,20.0,None,774705,774705,course-v1:DLI+S-FX-12+V1,None,None,None,None
5,DLI Training,2024-05-28T21:17:51.000Z,Getting Started with Deep Learning,Self-Paced Online,Attendee,Attended Live,6784220,20.0,None,772702,772702,course-v1:DLI+S-FX-01+V1,None,None,None,None
6,DLI Training,2024-06-19T01:55:54.000Z,Augment your LLM Using Retrieval Augmented Gen...,Self-Paced Online,Attendee,Attended Live,6878366,20.0,None,798402,798402,course-v1:DLI+S-FX-16+V1,None,None,None,None
7,DLI Training,2021-05-13T03:36:16.000Z,Data Augmentation and Segmentation with Genera...,Self-Paced Online,Attendee,Attended Live,1257703,20.0,None,307474,307474,course-v1:DLI+L-HX-09+V1-ZH,None,None,None,None
8,DLI Training,2024-06-19T03:30:41.000Z,Modeling Time Series Data with Recurrent Neura...,Self-Paced Online,Attendee,Attended Live,6878674,20.0,None,798483,798483,course-v1:DLI+L-FX-24+V1,None,None,None,None
9,DLI Training,2024-05-29T05:01:50.000Z,Building Real-Time Video AI Applications,Self-Paced Online,Attendee,Attended Live,6785420,20.0,None,773063,773063,course-v1:DLI+S-IV-01+V1,None,None,None,None


In [6]:

con.execute("""
SELECT column_name
FROM information_schema.columns
WHERE table_name = 'activity_clean'
ORDER BY ordinal_position
""").fetchdf()


,column_name
0,dev_contact
1,activity
2,activity_name
3,activity_type
4,activity_role
5,activity_attendance
6,activity_score
7,activity_date
8,activity_id
9,filepath


In [7]:

con.execute("SUMMARIZE activity_clean").fetchdf()


100% ▕██████████████████████████████████████▏ (00:00:02.66 elapsed)     


,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,dev_contact,VARCHAR,0003027cbcc9f926a435232e311b3a5ffcbb0d16668d12...,fffa9af220d19ab1ad50f0d2b1af409e4b0b0f732481d4...,6971468,NaN,NaN,NaN,NaN,NaN,69347526,0.00
1,activity,VARCHAR,Brev,"timeout-in-us = 4096 * (2**timeout)""",21,NaN,NaN,NaN,NaN,NaN,69347526,0.00
2,activity_name,VARCHAR,(COURSE ENDING ON 7/1/24) High-Performance Com...,（同時通訳付）Unlocking the Power of Agentic AI: Effe...,247536,NaN,NaN,NaN,NaN,NaN,69347526,7.12
3,activity_type,VARCHAR,API Catalog Hosted API,true,59,NaN,NaN,NaN,NaN,NaN,69347526,0.00
4,activity_role,VARCHAR,*Showstopper*,VivaTech Attendee,40,NaN,NaN,NaN,NaN,NaN,69347526,13.20
5,activity_attendance,VARCHAR,15,true,24,NaN,NaN,NaN,NaN,NaN,69347526,81.36
6,activity_score,DOUBLE,0.0,949763.0,16,3.4864210675072056,198.7994155581355,1.4549214275818465,3.0,3.0,69347526,1.31
7,activity_date,TIMESTAMP,2020-01-01 00:00:00,2026-03-12 17:24:45,10083528,2023-07-29 09:29:58.041171,NaN,2021-12-22 17:27:48.381268,2023-10-15 12:04:18.438245,2025-04-04 02:59:06.792796,69347526,0.00
8,activity_id,VARCHAR,--06jdo1meBBcl3t43Bw8twy5qo4b-WOFu9r_NPZ5JQ_20...,zzzKQIQtKhEv-aG9Dqw60fJX5IV8Rr8rrCGRIWBmS50_20...,51295433,NaN,NaN,NaN,NaN,NaN,69347526,0.00
9,filepath,VARCHAR,!git clone https://github.com/NVIDIA-AI-Bluepr...,zwn4k2csbt9r/nvcf/punctuation-riva,210473,NaN,NaN,NaN,NaN,NaN,69347526,22.16


### Contacts table baseline

In [8]:

con.execute("SELECT COUNT(*) AS row_count FROM contact_clean").fetchdf()


,row_count
0,9381490


In [9]:

con.execute("SELECT * FROM contact_raw LIMIT 30").fetchdf()


,developer_id,program_application_source,country,region,sub_region,zone,territory,organization_english_name,development_areas,other_development_areas,industry_segment_vertical,other_industry_segment_vertical,sub_industry_segment_vertical,fields_of_interest,other_fields_of_interest,first_program_application_date,account_id,account_name,last_activity_date,last_modified_date,created_date,wwfo_category,wwfo_target_list,account_industry_segment,account_source,account_type,devzone_last_login_date,organization_website,inception_id,first_activity_date,normalized_account_name,rdp_exit_date
0,9828293,dli,India,APAC,India,India,SOUTH ASIA,L&T Technology Services,Agentic AI / Generative AI;Data Science;Roboti...,null,AEC,null,null,null,null,2025-11-25T09:30:05.000Z,5768,L&T Technology Services,2025-11-25T09:32:29.000Z,2025-12-02T01:09:00.689Z,2025-11-25T09:30:05.200Z,null,null,null,Leadspace;GSI;NV Top 1000 Ent,Enterprise,null,ltts.com,null,2025-11-25T09:30:05.000Z,L&T Technology Services,null
1,9828383,dli,India,APAC,India,India,SOUTH ASIA,L&T Technology Services,Agentic AI / Generative AI;Data Science;Data C...,null,Automotive / Transportation,null,null,null,null,2025-11-25T09:41:30.000Z,5768,L&T Technology Services,2025-11-25T09:44:40.000Z,2025-12-10T01:09:46.643Z,2025-11-25T09:41:29.029Z,null,null,null,Leadspace;GSI;NV Top 1000 Ent,Enterprise,null,ltts.com,null,2025-11-25T09:41:30.000Z,L&T Technology Services,null
2,9828681,devzone,India,APAC,India,India,SOUTH ASIA,Dayananda Sagar Universiry,Agentic AI / Generative AI;AR / VR;Simulation ...,null,Other,null,null,null,null,2025-11-25T10:39:42.000Z,31153,Not Normalized,2026-03-10T00:00:00.000Z,2026-03-10T05:24:58.708Z,2025-11-25T10:39:42.694Z,null,null,null,Manual,NaN,2026-03-09T22:22:32.000Z,null,null,2025-11-25T00:00:00.000Z,Not Normalized,null
3,9829013,dli,India,APAC,India,India,SOUTH ASIA,LTTS,Developer Tools & Techniques;Agentic AI / Gene...,null,Other,null,null,null,null,2025-11-25T11:24:35.000Z,31153,Not Normalized,2025-11-27T12:29:09.000Z,2025-12-13T01:09:06.304Z,2025-11-25T11:24:33.415Z,null,null,null,Manual,NaN,null,null,null,2025-11-25T11:24:35.000Z,Not Normalized,null
4,9829125,devzone,India,APAC,India,India,SOUTH ASIA,Hexaware,Agentic AI / Generative AI;Data Center / Cloud,null,Cloud Services,null,null,null,null,2025-11-25T11:34:05.000Z,31153,Not Normalized,2025-12-04T12:27:20.000Z,2025-12-05T19:08:39.644Z,2025-11-25T11:34:04.830Z,null,null,null,Manual,NaN,2025-12-01T06:12:14.000Z,null,null,2025-11-25T11:34:05.000Z,Not Normalized,null
5,9829190,dli,India,APAC,India,India,SOUTH ASIA,L&T Technology Services,Agentic AI / Generative AI,null,Smart Cities / Spaces,null,null,null,null,2025-11-25T11:46:56.000Z,5768,L&T Technology Services,2025-12-03T10:09:20.000Z,2025-12-16T01:08:48.701Z,2025-11-25T11:46:56.248Z,null,null,null,Leadspace;GSI;NV Top 1000 Ent,Enterprise,null,ltts.com,null,2025-11-25T11:46:56.000Z,L&T Technology Services,null
6,9831912,dli,India,APAC,India,India,SOUTH ASIA,LTTS,Agentic AI / Generative AI,null,Other,null,null,null,null,2025-11-26T01:34:57.000Z,31153,Not Normalized,2025-12-01T02:38:25.000Z,2025-12-09T01:09:57.915Z,2025-11-26T01:34:56.652Z,null,null,null,Manual,NaN,null,null,null,2025-11-26T01:34:57.000Z,Not Normalized,null
7,9832663,null,India,APAC,India,India,SOUTH ASIA,Nokia,null,null,Other,null,null,null,null,null,2732,Nokia,2025-12-10T00:00:00.000Z,2025-12-10T18:16:17.016Z,2025-11-26T05:22:12.467Z,Telecommunications,SuperPOD 2019_P1,null,Fortune 500: 2020;WWFO; Fortune 500: 2021;Telc...,Enterprise,null,nokia.com,null,2025-12-10T00:00:00.000Z,Nokia,null
8,9833472,dli,India,APAC,India,India,SOUTH ASIA,MLRIT,Trustworthy AI / Cybersecurity;Data Science;Ag...,null,Academia / Education,null,null,null,null,2025-11-26T08:05:31.000Z,3297,Mlrit,2025-11-26T08:05:46.000Z,2025-11-27T01:09:11.830Z,2025-11-26T08:05:29.338Z,null,null,null,Leadspace,NaN,null,mlrit.ac.in,null,2025-11-26T08:05:31.000Z,Mlrit,null
9,9836934,dli,India,APAC,India,India,SOUTH ASIA,Tejas Net

In [10]:

con.execute("""
SELECT column_name
FROM information_schema.columns
WHERE table_name = 'contact_raw'
ORDER BY ordinal_position
""").fetchdf()


,column_name
0,developer_id
1,program_application_source
2,country
3,region
4,sub_region
5,zone
6,territory
7,organization_english_name
8,development_areas
9,other_development_areas


In [11]:

con.execute("SUMMARIZE contact_clean").fetchdf()


,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,developer_id,VARCHAR,000006aa196c7510ca3b4617e7e52d9527e0190e9152fb...,fffd473b66da19a680630437ec69aa8c7e450e2cec7848...,9168709,NaN,<NA>,NaN,NaN,NaN,9381490,0.00
1,program_application_source,VARCHAR,GTC,vcalliance,16,NaN,<NA>,NaN,NaN,NaN,9381490,4.76
2,country,VARCHAR,Afghanistan,null,270,NaN,<NA>,NaN,NaN,NaN,9381490,0.03
3,region,VARCHAR,APAC,null,4,NaN,<NA>,NaN,NaN,NaN,9381490,0.03
4,sub_region,VARCHAR,Africa,null,13,NaN,<NA>,NaN,NaN,NaN,9381490,0.03
5,zone,VARCHAR,ANZ,null,21,NaN,<NA>,NaN,NaN,NaN,9381490,0.03
6,territory,VARCHAR,AFRICA,null,18,NaN,<NA>,NaN,NaN,NaN,9381490,3.40
7,organization_english_name,VARCHAR,Babaground,󠀡󠀡,1805457,NaN,<NA>,NaN,NaN,NaN,9381490,0.15
8,development_areas,VARCHAR,AR / VR,null,104551,NaN,<NA>,NaN,NaN,NaN,9381490,0.06
9,other_development_areas,VARCHAR,AI & Digital Solutions;Analytics,？？？,22997,NaN,<NA>,NaN,NaN,NaN,9381490,4.78


### SDK Downloads table baseline

In [12]:

con.execute("SELECT COUNT(*) AS row_count FROM sdk_download_clean").fetchdf()


,row_count
0,93038213


In [13]:

con.execute("SELECT * FROM sdk_download_clean LIMIT 30").fetchdf()


,source,sdk_name,product_name,product_release,country,region,subregion,territory,zone,download_date,file_type,operating_system,os_distribution,architecture,kpi,download_count
0,PyPi,NCCL,nvidia-nccl-cu12,2.18.1,DZ,EMEA,Africa,AFRICA,Africa,2026-02-03,Package,Linux,None,None,1.0,7
1,PyPi,cuRAND,nvidia-curand-cu12,10.3.9.90,DZ,EMEA,Africa,AFRICA,Africa,2026-02-03,Package,Windows,None,None,1.0,2
2,PyPi,jax_products,jaxlib,0.4.30,DZ,EMEA,Africa,AFRICA,Africa,2026-02-03,Package,Darwin,None,None,1.0,4
3,PyPi,NVRTC,nvidia-cuda-nvrtc-cu12,12.9.86,DZ,EMEA,Africa,AFRICA,Africa,2026-02-03,Package,Linux,None,None,1.0,3
4,PyPi,cuRAND,nvidia-curand-cu12,10.3.10.19,DZ,EMEA,Africa,AFRICA,Africa,2026-02-03,Package,Linux,None,None,1.0,3
5,PyPi,jax_products,jax,0.9.0,LT,EMEA,Europe,POLAND_BALTICS,Eastern Europe,2026-02-03,Package,Linux,None,None,1.0,10
6,PyPi,openal_products,triton,2.1.0,LT,EMEA,Europe,POLAND_BALTICS,Eastern Europe,2026-02-03,Package,Linux,None,None,1.0,4
7,PyPi,listsonnx_products,onnxruntime,1.17.0,DZ,EMEA,Africa,AFRICA,Africa,2026-02-03,Package,Linux,None,None,1.0,2
8,PyPi,jax_products,jax,0.8.2,DZ,EMEA,Africa,AFRICA,Africa,2026-02-03,Package,Darwin,None,None,1.0,1
9,PyPi,tensorflow_products,tensorflow,2.16.1,DZ,EMEA,Africa,AFRICA,Africa,2026-02-03,Package,Linux,None,None,1.0,5


In [14]:

con.execute("""
SELECT column_name
FROM information_schema.columns
WHERE table_name = 'sdk_download_clean'
ORDER BY ordinal_position
""").fetchdf()


,column_name
0,source
1,sdk_name
2,product_name
3,product_release
4,country
5,region
6,subregion
7,territory
8,zone
9,download_date


In [15]:

con.execute("SUMMARIZE sdk_download_clean").fetchdf()


100% ▕██████████████████████████████████████▏ (00:00:02.60 elapsed)     


,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,source,VARCHAR,CDN,VSCODE,13,NaN,NaN,NaN,NaN,NaN,93038213,0.00
1,sdk_name,VARCHAR,3DGRUT,tensorflow_products,218,NaN,NaN,NaN,NaN,NaN,93038213,1.37
2,product_name,VARCHAR,(Linux) Acoustic Echo Cancellation,warp-lang,1183,NaN,NaN,NaN,NaN,NaN,93038213,0.00
3,product_release,VARCHAR,0.0,zjyu7g90o5jt/cosmos-1-0-diffusion-7b-text2worl...,11842,NaN,NaN,NaN,NaN,NaN,93038213,1.39
4,country,VARCHAR,AD,ZW,294,NaN,NaN,NaN,NaN,NaN,93038213,1.35
5,region,VARCHAR,APAC,NALA,3,NaN,NaN,NaN,NaN,NaN,93038213,1.35
6,subregion,VARCHAR,Africa,US & Canada,12,NaN,NaN,NaN,NaN,NaN,93038213,1.35
7,territory,VARCHAR,AFRICA,UK_NORDICS,17,NaN,NaN,NaN,NaN,NaN,93038213,42.43
8,zone,VARCHAR,ANZ,Western Europe,20,NaN,NaN,NaN,NaN,NaN,93038213,1.35
9,download_date,DATE,2020-01-01,2026-03-12,2293,2024-05-01 18:07:15.390675,NaN,2023-06-09,2024-10-10,2025-07-25,93038213,0.00



## 5. Record counts and uniqueness checks
These checks help identify duplicate inflation, key coverage issues, and whether row counts are aligned with the business meaning of each dataset.


In [16]:

queries = {
    "activity_rows": "SELECT COUNT(*) AS n FROM activity_clean",
    "activity_distinct_devs": "SELECT COUNT(DISTINCT dev_contact) AS n FROM activity_clean",
    "contact_rows": "SELECT COUNT(*) AS n FROM contact_clean",
    "contact_distinct_devs": "SELECT COUNT(DISTINCT developer_id) AS n FROM contact_clean",
    "sdk_rows": "SELECT COUNT(*) AS n FROM sdk_download_clean",
    "sdk_distinct_products": "SELECT COUNT(DISTINCT product_name) AS n FROM sdk_download_clean",
}

{key: con.execute(sql).fetchdf() for key, sql in queries.items()}


{'activity_rows':           n
 0  69347526,
 'activity_distinct_devs':          n
 0  7660278,
 'contact_rows':          n
 0  9381490,
 'contact_distinct_devs':          n
 0  9381490,
 'sdk_rows':           n
 0  93038213,
 'sdk_distinct_products':       n
 0  1158}

## 6. Schema-driven profiling by table

### 6.1 Activity table profiling

In [17]:
con.execute("SELECT activity, COUNT(*) AS cnt FROM activity_clean GROUP BY 1 ORDER BY cnt DESC").fetchdf()

,activity,cnt
0,DevZone Downloads,39742594
1,NGC Downloads,7720077
2,Dev Program Membership,6767607
3,Model API,6466203
4,DLI Training,1934108
5,On-Demand Views,1440586
6,Conf. Sessions Live,1312509
7,User Feedback,1238576
8,Forum Contributions,784059
9,Product Specific Comms,742469


In [18]:
con.execute("SELECT activity_type, COUNT(*) AS cnt FROM activity_clean GROUP BY 1 ORDER BY cnt DESC LIMIT 50").fetchdf()

,activity_type,cnt
0,Installer,36999321
1,Approved,6654555
2,API Catalog Hosted API,6466203
3,Container,5345238
4,Documentation,1791479
5,Self-Paced Online,1435932
6,NV On-Demand,1394454
7,Conference Session,1312509
8,Feedback Survey,1232012
9,Resource,1213352


In [19]:
con.execute("SELECT activity_attendance, COUNT(*) AS cnt FROM activity_clean GROUP BY 1 ORDER BY cnt DESC").fetchdf()

,activity_attendance,cnt
0,NaN,56421601
1,true,7551663
2,Attended Live,1960914
3,Attended Live;Registered,1699317
4,Attended On-Demand;Registered,1338780
5,Registered;Attended On-Demand,156936
6,Registered; Attended Live,126362
7,Registered;Attended Live,52684
8,Attended Live;Attended On-Demand;Registered,16653
9,Attended On-Demand,11986


In [20]:
con.execute("SELECT activity_role, COUNT(*) AS cnt FROM activity_clean GROUP BY 1 ORDER BY cnt DESC").fetchdf()

,activity_role,cnt
0,DZ4,20816265
1,Devzone,18926329
2,NaN,9150834
3,NGC,7720077
4,NVCF,6466203
5,Attendee,5247639
6,User,517729
7,Moderator,253783
8,Exhibitor,44318
9,Other,41124


In [21]:
con.execute("SELECT activity_score, COUNT(*) AS cnt FROM activity_clean GROUP BY 1 ORDER BY activity_score").fetchdf()

,activity_score,cnt
0,0.0,7304120
1,1.0,10136402
2,2.0,150807
3,3.0,40386220
4,4.0,569597
5,5.0,5173631
6,6.0,635828
7,8.0,143233
8,10.0,1907972
9,15.0,77497


In [22]:
con.execute("SELECT MIN(activity_date) AS min_date, MAX(activity_date) AS max_date FROM activity_clean").fetchdf()

,min_date,max_date
0,2020-01-01,2026-03-12 17:24:45



### 6.2 Contact table profiling
Use these fields to understand join coverage, organization quality, developer metadata, and potential segmentation dimensions.


In [23]:
con.execute("SELECT country, COUNT(*) AS cnt FROM contact_clean GROUP BY 1 ORDER BY cnt DESC LIMIT 25").fetchdf()

,country,cnt
0,United States,1688338
1,China,1645122
2,India,1111879
3,null,804111
4,"Korea, Republic of",305752
5,Japan,292502
6,Germany,238016
7,Taiwan,213672
8,United Kingdom,209229
9,Canada,159076


In [24]:
con.execute("SELECT region, COUNT(*) AS cnt FROM contact_clean GROUP BY 1 ORDER BY cnt DESC").fetchdf()

,region,cnt
0,APAC,4284392
1,NALA,2220135
2,EMEA,2070011
3,null,804111
4,NaN,2841


In [25]:
con.execute("SELECT zone, COUNT(*) AS cnt FROM contact_clean GROUP BY 1 ORDER BY cnt DESC").fetchdf()

,zone,cnt
0,US & Canada,1847414
1,China,1778896
2,India,1111879
3,Western Europe,974818
4,null,804111
5,SEA,372441
6,LatAm,355049
7,South Korea,305752
8,Japan,292502
9,Eastern Europe,241562


In [26]:
con.execute("SELECT account_type, COUNT(*) AS cnt FROM contact_clean GROUP BY 1 ORDER BY cnt DESC LIMIT 30").fetchdf()

,account_type,cnt
0,NaN,6404129
1,University,1980017
2,Enterprise,740459
3,Startup,215987
4,Enterprise;University,16654
5,Startup;University,12856
6,Enterprise;Startup,11388


In [27]:
con.execute("SELECT normalized_account_name, COUNT(*) AS cnt FROM contact_clean GROUP BY 1 ORDER BY cnt DESC LIMIT 30").fetchdf()

,normalized_account_name,cnt
0,Unclassified - Invalid,3165684
1,Not Normalized,2480837
2,NVIDIA,58054
3,Chandigarh University,34804
4,Peking University,29643
5,Tsinghua University,26077
6,Zhejiang University,21762
7,University of Electronic Science and Technolog...,19395
8,Shanghai Jiao Tong University,19114
9,Unclassified - Acronym,18356


In [28]:
con.execute("SELECT program_application_source, COUNT(*) AS cnt FROM contact_clean GROUP BY 1 ORDER BY cnt DESC").fetchdf()

,program_application_source,cnt
0,null,4006295
1,devzone,2806574
2,dli,1004037
3,gtc,448297
4,NaN,446092
5,api_catalog,391786
6,GTC Fall 2022,107396
7,GTC Spring 2022,89807
8,nod,46301
9,brev,10869


In [29]:
con.execute("SELECT MIN(first_activity_date) AS first_seen, MAX(last_activity_date) AS last_seen FROM contact_clean").fetchdf()

,first_seen,last_seen
0,2001-05-30,2026-04-15 12:46:21



### 6.3 SDK download profiling
This section profiles technology adoption behavior by source, product, operating system, and geography.


In [30]:
con.execute("SELECT source, COUNT(*) AS cnt FROM sdk_download_clean GROUP BY 1 ORDER BY cnt DESC").fetchdf()

,source,cnt
0,PyPi,71366571
1,CDN,16256643
2,NGC,4180625
3,CDN - PyPI,774093
4,GitHub,175664
5,CDN - APT,68791
6,CDN - Workbench,55483
7,Conda,47921
8,CDN - Azure,39300
9,DOCKERHUB,38529


In [31]:
con.execute("SELECT sdk_name, COUNT(*) AS cnt FROM sdk_download_clean GROUP BY 1 ORDER BY cnt DESC LIMIT 50").fetchdf()

,sdk_name,cnt
0,tensorflow_products,12104248
1,listsonnx_products,7631106
2,CUDA Toolkit,7350940
3,jax_products,7045281
4,lightning_products,7006933
5,pytorch_products,5729419
6,cuDNN,3546075
7,Triton,2553631
8,Inference Serving,2323808
9,NCCL,2297304


In [32]:
con.execute("SELECT product_name, COUNT(*) AS cnt FROM sdk_download_clean GROUP BY 1 ORDER BY cnt DESC LIMIT 50").fetchdf()

,product_name,cnt
0,tensorflow,9207348
1,pytorch-lightning,5702569
2,torch,5607273
3,CUDA Toolkit,5404957
4,jax,4926955
5,onnxruntime,2905723
6,onnx,2543826
7,cuDNN,2108422
8,jaxlib,2070727
9,vllm,1710983


In [33]:
con.execute("SELECT file_type, COUNT(*) AS cnt FROM sdk_download_clean GROUP BY 1 ORDER BY cnt DESC").fetchdf()

,file_type,cnt
0,Package,71442184
1,Installer - KPI,9508317
2,Container,4180223
3,Installer,1842700
4,Documentation,1287318
5,NaN,1199133
6,whl,1089181
7,Installer-KPI,909504
8,Image,235093
9,Installer KPI,215591


In [34]:
con.execute("SELECT operating_system, COUNT(*) AS cnt FROM sdk_download_clean GROUP BY 1 ORDER BY cnt DESC").fetchdf()

,operating_system,cnt
0,Linux,37102242
1,Windows,24154098
2,NaN,17946361
3,Darwin,8695657
4,Linux x86,2947007
...,...,...
209,Nto,1
210,MINGW32_NT-10.0-19043,1
211,MINGW64_NT-10.0-WOW,1
212,CYGWIN_NT-6.3-9600-WOW64,1


In [35]:
con.execute("SELECT architecture, COUNT(*) AS cnt FROM sdk_download_clean GROUP BY 1 ORDER BY cnt DESC").fetchdf()

,architecture,cnt
0,NaN,88142755
1,x86_64,3979916
2,ARM,506941
3,aarch64,149427
4,arm64-sbsa,67088
5,PowerPC,60500
6,x86,35593
7,Amd64,34124
8,aarch64-jetson,30890
9,Universal,10735


In [36]:
con.execute("SELECT MIN(download_date) AS first_download, MAX(download_date) AS last_download FROM sdk_download_clean").fetchdf()

,first_download,last_download
0,2020-01-01,2026-03-12



## 7. Data quality checks
Before drawing conclusions, quantify nulls, duplicates, invalid values, and join failures.


In [37]:

quality_queries = {
    "activity_nulls": """
        SELECT
            COUNT(*) AS total_rows,
            SUM(CASE WHEN dev_contact IS NULL THEN 1 ELSE 0 END) AS null_dev_contact,
            SUM(CASE WHEN activity_date IS NULL THEN 1 ELSE 0 END) AS null_activity_date,
            SUM(CASE WHEN activity IS NULL THEN 1 ELSE 0 END) AS null_activity,
            SUM(CASE WHEN activity_score IS NULL THEN 1 ELSE 0 END) AS null_activity_score,
            SUM(CASE WHEN activity_id IS NULL THEN 1 ELSE 0 END) AS null_activity_id
        FROM activity_clean
    """,
    "contact_nulls": """
        SELECT
            COUNT(*) AS total_rows,
            SUM(CASE WHEN developer_id IS NULL THEN 1 ELSE 0 END) AS null_developer_id,
            SUM(CASE WHEN country IS NULL THEN 1 ELSE 0 END) AS null_country,
            SUM(CASE WHEN region IS NULL THEN 1 ELSE 0 END) AS null_region,
            SUM(CASE WHEN normalized_account_name IS NULL THEN 1 ELSE 0 END) AS null_normalized_account_name,
            SUM(CASE WHEN first_activity_date IS NULL THEN 1 ELSE 0 END) AS null_first_activity_date
        FROM contact_clean
    """,
    "sdk_nulls": """
        SELECT
            COUNT(*) AS total_rows,
            SUM(CASE WHEN source IS NULL THEN 1 ELSE 0 END) AS null_source,
            SUM(CASE WHEN product_name IS NULL THEN 1 ELSE 0 END) AS null_productname,
            SUM(CASE WHEN download_date IS NULL THEN 1 ELSE 0 END) AS null_downloaddate,
            SUM(CASE WHEN country IS NULL THEN 1 ELSE 0 END) AS null_country
        FROM sdk_download_clean
    """
}

{key: con.execute(sql).fetchdf() for key, sql in quality_queries.items()}


{'activity_nulls':    total_rows  null_dev_contact  null_activity_date  null_activity  \
 0    69347526              25.0                 3.0            0.0   
 
    null_activity_score  null_activity_id  
 0             906358.0              20.0  ,
 'contact_nulls':    total_rows  null_developer_id  null_country  null_region  \
 0     9381490                0.0        2841.0       2841.0   
 
    null_normalized_account_name  null_first_activity_date  
 0                           0.0                 1600085.0  ,
 'sdk_nulls':    total_rows  null_source  null_productname  null_downloaddate  null_country
 0    93038213          0.0               0.0                0.0     1254777.0}

In [38]:

con.execute("""
SELECT activity_id, COUNT(*) AS cnt
FROM activity_clean
WHERE activity_id IS NOT NULL
GROUP BY 1
HAVING COUNT(*) > 1
ORDER BY cnt DESC
LIMIT 50
""").fetchdf()


100% ▕██████████████████████████████████████▏ (00:00:02.10 elapsed)     


,activity_id,cnt
0,SURVEY_185-6182646,398
1,SURVEY_185-6475434,172
2,SURVEY_185-6228214,159
3,hzqaa2mdg-8519393,146
4,SURVEY_185-539824,137
5,SURVEY_185-4790429,118
6,SURVEY_185-2544411,115
7,r5qmxaqcv-3199690,105
8,SURVEY_185-6522069,81
9,v16whbpr2-9091179,80


In [39]:

con.execute("""
SELECT pk1, COUNT(*) AS cnt
FROM activity_clean
WHERE pk1 IS NOT NULL
GROUP BY 1
HAVING COUNT(*) > 1
ORDER BY cnt DESC
LIMIT 50
""").fetchdf()


,pk1,cnt
0,SURVEY_185-6182646,398
1,SURVEY_185-6475434,172
2,SURVEY_185-6228214,159
3,hzqaa2mdg-8519393,146
4,SURVEY_185-539824,137
5,SURVEY_185-4790429,118
6,SURVEY_185-2544411,115
7,r5qmxaqcv-3199690,105
8,SURVEY_185-6522069,81
9,v16whbpr2-9091179,80


In [40]:

con.execute("""
SELECT developer_id, COUNT(*) AS cnt
FROM contact_clean
GROUP BY 1
HAVING COUNT(*) > 1
ORDER BY cnt DESC
LIMIT 50
""").fetchdf()


,developer_id,cnt


In [41]:

con.execute("""
SELECT country, COUNT(DISTINCT region) AS region_count
FROM contact_clean
WHERE country IS NOT NULL
GROUP BY 1
HAVING COUNT(DISTINCT region) > 1
ORDER BY region_count DESC, country
""").fetchdf()


,country,region_count



## 8. Join coverage and entity alignment
This section determines how well the three tables connect at the developer level and where the data model has blind spots.


In [42]:

con.execute("""
SELECT
    COUNT(DISTINCT a.dev_contact) AS activity_developers,
    COUNT(DISTINCT c.developer_id) AS contact_developers,
    COUNT(DISTINCT a.dev_contact) FILTER (WHERE c.developer_id IS NOT NULL) AS activity_devs_found_in_contacts,
    COUNT(DISTINCT a.dev_contact) FILTER (WHERE c.developer_id IS NULL) AS activity_devs_missing_in_contacts
FROM activity_clean a
LEFT JOIN contact_clean c
    ON a.dev_contact = c.developer_id
""").fetchdf()


100% ▕██████████████████████████████████████▏ (00:00:02.08 elapsed)     


,activity_developers,contact_developers,activity_devs_found_in_contacts,activity_devs_missing_in_contacts
0,7660278,7660260,7660260,18



If the SDK table does not contain a developer identifier, treat it as product and geography level adoption evidence rather than person-level adoption. If it does contain one in your local version, replace the placeholder column name below and run the join coverage test.


In [43]:

# Replace sdk_dev_key with the correct developer join field if one exists in your local sdk_download_clean table.
# Example: sdk_dev_key = "developer_id"
sdk_dev_key = None

if sdk_dev_key:
    query = f"""
    SELECT
        COUNT(DISTINCT {sdk_dev_key}) AS sdk_developers,
        COUNT(DISTINCT {sdk_dev_key}) FILTER (WHERE c.developer_id IS NOT NULL) AS sdk_devs_found_in_contacts,
        COUNT(DISTINCT {sdk_dev_key}) FILTER (WHERE c.developer_id IS NULL) AS sdk_devs_missing_in_contacts
    FROM sdk_download_clean s
    LEFT JOIN contact_clean c
        ON s.{sdk_dev_key} = c.developer_id
    """
    display(con.execute(query).fetchdf())
else:
    print("No developer-level join field configured for sdk_download_clean.")


No developer-level join field configured for sdk_download_clean.



## 9. Temporal consistency and developer journey checks
These checks look for timeline contradictions and early signs of journey funnel structure.


In [44]:

con.execute("""
SELECT COUNT(*) AS activities_before_first_activity
FROM activity_clean a
JOIN contact_clean c
    ON a.dev_contact = c.developer_id
WHERE a.activity_date < CAST(c.first_activity_date AS DATE)
""").fetchdf()


,activities_before_first_activity
0,36


In [45]:

con.execute("""
SELECT COUNT(*) AS activities_after_last_activity
FROM activity_clean a
JOIN contact_clean c
    ON a.dev_contact = c.developer_id
WHERE c.last_activity_date IS NOT NULL
  AND a.activity_date > CAST(c.last_activity_date AS DATE)
""").fetchdf()


,activities_after_last_activity
0,6527407


In [46]:

con.execute("""
SELECT
    DATE_TRUNC('month', activity_date) AS month,
    COUNT(*) AS activity_rows,
    COUNT(DISTINCT dev_contact) AS active_developers
FROM activity_clean
GROUP BY 1
ORDER BY 1
""").fetchdf()


,month,activity_rows,active_developers
0,2020-01-01,583859,86164
1,2020-02-01,562155,87388
2,2020-03-01,642292,106830
3,2020-04-01,679316,119127
4,2020-05-01,722268,119708
5,2020-06-01,729770,114933
6,2020-07-01,774524,111654
7,2020-08-01,651875,103609
8,2020-09-01,699645,115548
9,2020-10-01,913007,137373



## 10. Activity intensity and developer-level aggregation
A developer-level feature table is the bridge between exploratory analysis and segmentation.


In [47]:

developer_features_query = """
WITH activity_features AS (
    SELECT
        dev_contact AS developer_id,
        COUNT(*) AS activity_rows,
        COUNT(DISTINCT activity) AS distinct_activity_categories,
        COUNT(DISTINCT activity_name) AS distinct_activity_names,
        SUM(COALESCE(activity_score, 0)) AS total_activity_score,
        AVG(COALESCE(activity_score, 0)) AS avg_activity_score,
        MIN(activity_date) AS first_activity_date_from_events,
        MAX(activity_date) AS last_activity_date_from_events,
        COUNT(*) FILTER (WHERE activity = 'Webinars') AS webinar_rows,
        COUNT(*) FILTER (WHERE activity = 'DLI Training') AS dli_rows,
        COUNT(*) FILTER (WHERE activity = 'Hackathon') AS hackathon_rows,
        COUNT(*) FILTER (WHERE activity = 'Forum Contributions') AS forum_rows,
        COUNT(*) FILTER (WHERE activity = 'Hosted API') AS hosted_api_rows,
        COUNT(*) FILTER (WHERE activity = 'DevZone Downloads') AS devzone_download_rows,
        COUNT(*) FILTER (WHERE activity = 'NGC Downloads') AS ngc_download_rows
    FROM activity_clean
    GROUP BY 1
)
SELECT
    af.*, 
    c.country,
    c.region,
    c.zone,
    c.account_type,
    c.program_application_source,
    c.normalized_account_name,
    c.first_activity_date,
    c.last_activity_date
FROM activity_features af
LEFT JOIN contact_clean c
    ON af.developer_id = c.developer_id
"""

dev_features = con.execute(developer_features_query).fetchdf()
dev_features.head()


100% ▕██████████████████████████████████████▏ (00:00:16.16 elapsed)     
 73% ▕███████████████████████████▋          ▏ (~28 seconds remaining)   

,developer_id,activity_rows,distinct_activity_categories,distinct_activity_names,total_activity_score,avg_activity_score,first_activity_date_from_events,last_activity_date_from_events,webinar_rows,dli_rows,hackathon_rows,forum_rows,hosted_api_rows,devzone_download_rows,ngc_download_rows,country,region,zone,account_type,program_application_source,normalized_account_name,first_activity_date,last_activity_date
0,889124,4,4,4,27.0,6.750000,2022-05-25 00:00:00,2025-11-19 08:37:48,0,1,0,0,0,1,0,Israel,EMEA,Israel,NaN,null,Arbe Robotics,2018-08-13 00:00:00,2025-11-19 08:37:48
1,3184433,34,4,5,172.0,5.058824,2023-01-18 01:45:33,2026-01-19 00:00:00,0,2,0,0,0,29,0,Taiwan,APAC,Taiwan,NaN,dli,Not Normalized,2023-01-18 01:45:33,2026-01-19 00:00:00
2,6112715,8,1,4,320.0,40.000000,2022-04-07 05:49:05,2022-04-28 05:41:24,0,8,0,0,0,0,0,China,APAC,China,Enterprise,null,LY Corporation,2022-04-07 05:49:05,2022-04-28 05:41:24
3,5887378,73,7,55,1008.0,13.808219,2020-07-12 00:00:00,2026-01-22 07:49:57,4,28,0,0,0,14,0,Singapore,APAC,SEA,Enterprise,NaN,Edwards Lifesciences,2014-12-30 00:00:00,2026-01-22 07:49:57
4,9371020,3,2,3,41.0,13.666667,2025-08-13 04:35:30,2025-08-13 04:38:11,0,2,0,0,0,0,0,India,APAC,India,University,dli,Symbiosis International University,2025-08-13 04:35:30,2025-08-13 04:38:11


In [48]:
dev_features.describe(include="all").transpose()

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
developer_id,7660278,7660278,889124,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
activity_rows,7660279.0,NaN,NaN,NaN,9.052872,1.0,1.0,2.0,4.0,2027375.0,800.05119
distinct_activity_categories,7660279.0,NaN,NaN,NaN,1.951847,1.0,1.0,2.0,2.0,16.0,0.878217
distinct_activity_names,7660279.0,NaN,NaN,NaN,2.809851,0.0,1.0,2.0,3.0,10748.0,11.012568
total_activity_score,7660279.0,NaN,NaN,NaN,31.149613,0.0,3.0,7.0,20.0,10015132.0,4064.747477
avg_activity_score,7660279.0,NaN,NaN,NaN,3.963967,0.0,1.75,2.333333,3.0,113915.24,41.44732
first_activity_date_from_events,7660279,NaN,NaN,NaN,2023-08-11 17:49:48.343630,2020-01-01 00:00:00,2022-02-18 00:00:00,2024-01-04 00:00:00,2025-03-10 10:56:43,2026-03-12 17:24:45,NaN
last_activity_date_from_events,7660279,NaN,NaN,NaN,2023-12-16 22:31:47.008409,2020-01-01 00:00:00,2022-09-13 11:36:43,2024-05-15 08:46:05,2025-05-13 05:35:40.500000,2026-03-12 17:24:45,NaN
webinar_rows,7660279.0,NaN,NaN,NaN,0.041775,0.0,0.0,0.0,0.0,132.0,0.402579
dli_rows,7660279.0,NaN,NaN,NaN,0.252485,0.0,0.0,0.0,0.0,228.0,1.048795



## 11. Investigating weird patterns and anomalies
This section directly targets unusual behavior that may become high-value business insights.


In [49]:

con.execute("""
SELECT activity, AVG(COALESCE(activity_score, 0)) AS avg_score, MIN(activity_score) AS min_score, MAX(activity_score) AS max_score
FROM activity_clean
GROUP BY 1
ORDER BY avg_score DESC
""").fetchdf()


,activity,avg_score,min_score,max_score
0,I want to know how this feature can be manipul...,949763.000000,949763.0,949763.0
1,"timeout-in-us = 4096 * (2**timeout)""",949156.000000,949156.0,949156.0
2,during soft link creation in setting up the NV...,948905.000000,948905.0,948905.0
3,DLI Training,25.151481,20.0,40.0
4,Conf. Sessions Live,10.804406,10.0,100.0
5,Conference,9.985589,3.0,100.0
6,Webinars,7.099192,0.0,10.0
7,Other Events,5.737479,3.0,10.0
8,NGC Downloads,4.247388,1.0,6.0
9,On-Demand Views,3.713054,0.0,100.0


In [50]:

con.execute("""
SELECT dev_contact, COUNT(*) AS rows_per_dev, SUM(COALESCE(activity_score, 0)) AS total_score
FROM activity_clean
GROUP BY 1
ORDER BY total_score DESC, rows_per_dev DESC
LIMIT 50
""").fetchdf()


,dev_contact,rows_per_dev,total_score
0,5903679,2027375,10015132.0
1,5886356,789400,3946993.0
2,NaN,25,2847881.0
3,3931685,187095,934975.0
4,2028858,142980,714100.0
5,1303597,151947,449262.0
6,6125204,68992,344958.0
7,4833930,55061,275414.0
8,225774,53677,267525.0
9,6309430,121950,233315.0


In [51]:

con.execute("""
SELECT normalized_account_name, COUNT(*) AS dev_count
FROM contact_clean
WHERE normalized_account_name IS NOT NULL
GROUP BY 1
ORDER BY dev_count DESC
LIMIT 50
""").fetchdf()


,normalized_account_name,dev_count
0,Unclassified - Invalid,3165684
1,Not Normalized,2480837
2,NVIDIA,58054
3,Chandigarh University,34804
4,Peking University,29643
5,Tsinghua University,26077
6,Zhejiang University,21762
7,University of Electronic Science and Technolog...,19395
8,Shanghai Jiao Tong University,19114
9,Unclassified - Acronym,18356


In [52]:

con.execute("""
SELECT country, region, sub_region, territory, zone, COUNT(*) AS rows
FROM contact_clean
GROUP BY 1,2,3,4,5
ORDER BY rows DESC
LIMIT 50
""").fetchdf()


,country,region,sub_region,territory,zone,rows
0,United States,NALA,US & Canada,null,US & Canada,1596732
1,China,APAC,China,null,China,1539154
2,India,APAC,India,SOUTH ASIA,India,1111879
3,null,null,null,null,null,804111
4,"Korea, Republic of",APAC,South Korea,null,South Korea,278575
5,Japan,APAC,Japan,null,Japan,262327
6,Germany,EMEA,Europe,DACH,Western Europe,238016
7,United Kingdom,EMEA,Europe,UK_NORDICS,Western Europe,209229
8,Taiwan,APAC,Taiwan,null,Taiwan,197293
9,Canada,NALA,US & Canada,null,US & Canada,147395



### Suggested anomaly follow-ups
Use the observations above to document findings such as:

- developers with very high activity scores but narrow activity diversity
- silent adopters who appear only in download behavior
- countries mapped inconsistently to regions or zones
- organization fields with excessive "Not Normalized" values
- duplicated activity records caused by event form design or source extraction logic



## 12. Segmentation analysis roadmap
The project can support several segmentation lenses. Start simple, then combine them.



### Segmentation families

**Engagement depth segmentation**
- low, medium, high total activity score
- number of distinct activity categories
- recency and frequency of touchpoints

**Journey type segmentation**
- event-heavy developers
- training-heavy developers
- community contributors
- product download oriented developers
- one-time explorers versus repeat engagers

**Account and geography segmentation**
- enterprise versus startup versus university
- region and zone differences
- normalized versus non-normalized organizations

**Adoption propensity segmentation**
- activity-rich developers with product download signals
- activity-rich developers with no clear adoption signals
- low-touch developers with concentrated product behavior


In [53]:

con.execute("""
WITH scored AS (
    SELECT
        dev_contact,
        SUM(COALESCE(activity_score, 0)) AS total_score,
        COUNT(*) AS total_rows,
        COUNT(DISTINCT activity) AS distinct_activities
    FROM activity_clean
    GROUP BY 1
), bucketed AS (
    SELECT
        *,
        NTILE(4) OVER (ORDER BY total_score) AS score_quartile
    FROM scored
)
SELECT score_quartile,
       COUNT(*) AS developers,
       AVG(total_score) AS avg_total_score,
       AVG(total_rows) AS avg_rows,
       AVG(distinct_activities) AS avg_distinct_activities
FROM bucketed
GROUP BY 1
ORDER BY 1
""").fetchdf()


100% ▕██████████████████████████████████████▏ (00:00:02.01 elapsed)     


,score_quartile,developers,avg_total_score,avg_rows,avg_distinct_activities
0,1,1915070,1.184688,4.000105,1.161261
1,2,1915070,4.024865,2.027993,1.829085
2,3,1915070,10.639530,3.989583,2.276549
3,4,1915069,108.749412,26.193817,2.540493



## 13. Asset impact analysis candidates
These analyses move from description toward causal or quasi-causal business storytelling.



Potential extensions:
1. Measure whether selected assets are associated with increased downstream activity score in the next 30, 60, or 90 days.
2. Compare behavior before and after a webinar, DLI training, or hackathon.
3. Identify which activity names are most associated with follow-up multi-touch engagement.
4. Build developer-level event sequences and common journey paths.


In [54]:

# Example starter query for pre/post asset analysis.
asset_name = None  # Replace with an activity_name of interest.

if asset_name:
    query = f"""
    WITH asset_users AS (
        SELECT dev_contact, MIN(activity_date) AS first_asset_date
        FROM activity_clean
        WHERE activity_name = '{asset_name}'
        GROUP BY 1
    )
    SELECT
        COUNT(DISTINCT a.dev_contact) AS developers,
        AVG(CASE WHEN a.activity_date < u.first_asset_date THEN 1 ELSE 0 END) AS pre_asset_row_share,
        AVG(CASE WHEN a.activity_date > u.first_asset_date THEN 1 ELSE 0 END) AS post_asset_row_share
    FROM activity_clean a
    JOIN asset_users u
        ON a.dev_contact = u.dev_contact
    """
    display(con.execute(query).fetchdf())
else:
    print("Set asset_name to evaluate a specific asset.")


Set asset_name to evaluate a specific asset.


In [56]:
con.close()


## 14. Data enrichment plan ideas
This section is for the final deliverable around missing context and external supplementation.



Possible enrichment opportunities:
- standardize organization names and websites using external company matching logic
- enrich country and organization with market maturity or industry metadata
- classify activity names into a cleaner asset taxonomy
- enrich sdk_name or PRODUCTNAME into product families
- identify developer lifecycle stage from first and last activity windows
